# 09 — Task A vs Task B: does leakage matter, and where? (issue #17)

Notebooks 07 and 10 ran the *same* analysis code (`ehrgraph.netstats.task_analysis`) on the two locked tasks. This notebook puts them side by side:

| | Task A: Long Length of Stay | Task B: Hypertension |
|---|---|---|
| Predicts | Will this stay be long? | First hypertension diagnosis within a year? |
| Predicted at | End of the admission day | End of the discharge day |
| Is the outcome itself a diagnosis code? | **No** | **Yes** |

The last row is why the comparison matters. Our leaky graph lets each node see diagnoses from its whole record, including the future. For Task B, the future record can contain the answer itself.

**This notebook reads only `results/aggregate/*.csv`.** It needs no patient data, so anyone can run it, and so could CI.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from ehrgraph.viz import network_figures as nf  # noqa: E402

results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
TASKS = {"guo_los": "Task A: Long Length of Stay", "new_hypertension": "Task B: Hypertension"}
df = pd.concat(
    [pd.read_csv(results_dir / f) for f in ("network_stats_task_a.csv", "network_stats_task_b.csv")]
)
df = df[df["seed"].isna()]  # ensemble summaries and real graphs; per-seed null rows excluded
nf.apply_style()


def value(task, condition, metric, col="value"):
    hit = df[(df["task"] == task) & (df["condition"] == condition) & (df["metric"] == metric)]
    return float(hit[col].iloc[0]) if len(hit) else np.nan


print(df.groupby("task")["metric"].count())

## The comparison in one table

Every row is one number per task and graph. Null values are the mean of 10 rewired graphs.

In [ ]:
METRICS = {
    "prevalence": "Outcome base rate",
    "class_insensitive_homophily": "Homophily (0 = random)",
    "class_insensitive_homophily_excl_same_patient": "Homophily, without same-patient links",
    "dirichlet_energy_clmbr": "Dirichlet energy (lower = neighbors alike)",
    "leiden_outcome_eta_squared": "Outcome variance explained by community (η²)",
    "leiden_modularity": "Modularity",
    "transitivity": "Transitivity (closed triangles)",
    "largest_cc_fraction": "Share in largest piece",
    "same_patient_edge_fraction": "Share of links that are same-patient",
    "max_single_code_outcome_gap": "Largest single-code positive-negative gap",
}
rows = []
for metric, label in METRICS.items():
    for task, task_label in TASKS.items():
        conds = ("reference",) if metric == "prevalence" else ("safe", "leaky", "null_model")
        rows.append(
            {"metric": label, "task": task_label}
            | {c if c != "reference" else "safe": value(task, c, metric) for c in conds}
        )
table = pd.DataFrame(rows).set_index(["metric", "task"])
table["leaky − safe"] = table["leaky"] - table["safe"]
table.round(3)

## Where the two tasks diverge

Each panel is one measure. Dots are the safe and leaky graphs, and the bar is the null ensemble's 95% range. The question for each panel: does the leaky graph pull away from the safe one, and in which task?

In [ ]:
PANELS = [
    ("class_insensitive_homophily", "Homophily"),
    ("class_insensitive_homophily_excl_same_patient", "Homophily, no same-patient links"),
    ("dirichlet_energy_clmbr", "Dirichlet energy (CLMBR)"),
    ("leiden_outcome_eta_squared", "Outcome explained by community (η²)"),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (metric, title) in zip(axes, PANELS, strict=True):
    for x, task in enumerate(TASKS):
        lo, hi = (
            value(task, "null_model", metric, "ci_low"),
            value(task, "null_model", metric, "ci_high"),
        )
        ax.vlines(x, lo, hi, color=nf.COLORS["null"], lw=8, alpha=0.35)
        ax.plot(x, value(task, "null_model", metric), "_", color=nf.COLORS["null"], ms=14, mew=2)
        for dx, cond in ((-0.12, "safe"), (0.12, "leaky")):
            v = value(task, cond, metric)
            ax.plot(x + dx, v, "o", color=nf.COLORS[cond], ms=9, label=cond if x == 0 else None)
            ax.annotate(
                f"{v:.3f}",
                (x + dx, v),
                xytext=(0, 8),
                textcoords="offset points",
                ha="center",
                color=nf.INK_2,
                fontsize=8,
            )
    ax.set_xticks([0, 1], ["Task A\nLength of stay", "Task B\nHypertension"])
    ax.set_xlim(-0.6, 1.6)
    ax.grid(axis="x", visible=False)
    ax.set_title(title, fontsize=11)
axes[0].plot([], [], "_", color=nf.COLORS["null"], ms=14, mew=2, label="null (95% range)")
axes[0].legend(loc="upper left", fontsize=8)
fig.tight_layout()
nf.save(fig, figures_dir / "09_cross_task_comparison.png")
plt.show()

## Does the future record contain the answer?

For each task, the largest gap between the share of positive and of negative nodes whose diagnosis vector contains any one code. A gap near 1 would mean a single code nearly *is* the label.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
width = 0.36
for i, cond in enumerate(("safe", "leaky")):
    vals = [value(t, cond, "max_single_code_outcome_gap") for t in TASKS]
    bars = ax.bar(
        np.arange(2) + (i - 0.5) * width, vals, width - 0.04, color=nf.COLORS[cond], label=cond
    )
    for b in bars:
        ax.annotate(
            f"{b.get_height():.2f}",
            (b.get_x() + b.get_width() / 2, b.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            color=nf.INK_2,
        )
ax.set_xticks([0, 1], list(TASKS.values()))
ax.set_ylim(0, 1)
ax.set_ylabel("positive rate − negative rate")
ax.set_title("Strongest single-code signal for the outcome")
ax.grid(axis="x", visible=False)
ax.legend()
fig.tight_layout()
nf.save(fig, figures_dir / "09_single_code_leakage.png")
plt.show()

## Findings

**The answer to "does leakage matter?" depends on whether the outcome is written in the record.**

| | Task A: Long Length of Stay | Task B: Hypertension |
|---|---|---|
| Strongest single code, leaky vectors | 0.16 gap (abnormal lung imaging) | **0.62 gap (essential hypertension itself)** |
| Homophily, leaky − safe | −0.008 | **+0.050** |
| Homophily without same-patient links, leaky − safe | +0.004 | **+0.057** |
| Outcome explained by community (η²), leaky − safe | +0.026 | **+0.046** |
| Dirichlet energy, leaky − safe | +0.002 | −0.004 |

1. **Task A (long stay): leakage changes the graph's shape, not its outcome signal.** Length of stay is not a diagnosis, so the future record holds no answer key, and the links leakage swaps in are slightly worse than the ones it removes.
2. **Task B (hypertension): leakage measurably improves outcome alignment.** The leaky vectors contain the diagnosis being predicted, and homophily without same-patient links rises 42% (0.136 → 0.193). This is the task where the paper's leakage warning is most likely to show up in Project 2's predictions.
3. **Both real graphs are far from random in both tasks.** Homophily is 0.21–0.33 against a null of 0.004–0.011. Dirichlet energy is 0.65–0.69 against 0.95. Modularity is 0.93–0.98 against 0.56. The structure is real, and it is tied to the outcome in both tasks.
4. **Same-patient links behave in opposite ways.** They are 31–40% of all links in both tasks. In Task A they *dilute* homophily, since a patient's stays vary in length. In Task B they *carry half of it*, since hypertension status persists, and they agree 96% of the time. Any homophily-vs-gain claim must report both versions.
5. **CLMBR alignment barely moves with leakage** (energy differences under 0.005 in both tasks). Leakage changes *which* patients are neighbors, but neighbors look equally alike to CLMBR either way. Any leakage-driven gain in Project 2 would therefore come through the labels the graph lets a model borrow, not through smoother fingerprints.

**For the Project 1 report**, this is the headline comparison. **For Project 2**, it sets a concrete expectation: the leaky-vs-safe gap should be near zero for long stay and visible for hypertension. If long stay shows a large leaky gain anyway, something else is going on and needs investigating before it is reported.

### Done when
- [x] #17: both tasks compared through identical code, from aggregate CSVs only (`09_cross_task_comparison.png`, `09_single_code_leakage.png`)